## Evaluating small language models for tutoring responses
### Based on  latency, quality of response, tutoring pov, size
Llama3.2-1B, Qwen2.5-1.5B, DeepSeeek-R1-1.5B, SmolLM2-1.7B, Phi-3.5-Mini-3.8B, Gemma3-4B, Ministral-3-3B-Instruct-2512, OpenELM-1_1B


### Prerequisite

In [ ]:
# !pip install huggingface_hub  # Uncomment if not installed

import os
from huggingface_hub import hf_hub_download

models = {
    "llama3.2-1b": ("bartowski/Llama-3.2-1B-Instruct-GGUF", "Llama-3.2-1B-Instruct-Q4_K_M.gguf"),
    "qwen2.5-1.5b": ("Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-q4_k_m.gguf"),
    "deepseek-r1-1.5b": ("unsloth/DeepSeek-R1-Distill-Qwen-1.5B-GGUF", "DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf"),
    "smollm2-1.7b": ("HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF", "SmolLM2-1.7B-Instruct-Q4_K_M.gguf"),
    "phi3.5-mini": ("bartowski/Phi-3.5-mini-3.8B-ArliAI-RPMax-v1.1-GGUF", "Phi-3.5-mini-3.8B-ArliAI-RPMax-v1.1-Q4_K_M.gguf"),
    "openelm-1.1b": ("mradermacher/OpenELM-1_1B-GGUF", "OpenELM-1_1B.Q4_K_M.gguf"),
    "ministral3.3-3b": ("unsloth/Ministral-3-3B-Instruct-2512-GGUF", "Ministral-3-3B-Instruct-2512-Q4_K_M.gguf"),
    "gemma3-4b": ("unsloth/gemma-3-4b-it-GGUF", "gemma-3-4b-it-Q4_K_M.gguf")
}

target_dir = os.path.abspath("./model") 
os.makedirs(target_dir, exist_ok=True)

print(f"Target directory: {target_dir}")

for nickname, (repo_id, filename) in models.items():
    print(f"--- Downloading {nickname} ---")
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=target_dir,
            local_dir_use_symlinks=False  # Downloads the actual file instead of a link
        )
        print(f"Successfully downloaded to: {path}\n")
    except Exception as e:
        print(f"Error downloading {nickname}: {e}\n")


In [ ]:
import subprocess
import time
import os

# Start 'ollama serve' in the background
# This redirected output to null to keep the notebook clean, 
# but you can point it to a file if you want to see logs.
with open(os.devnull, 'wb') as devnull:
    process = subprocess.Popen(
        ['ollama', 'serve'], 
        stdout=devnull, 
        stderr=devnull
    )

# Wait a moment for the server to initialize
time.sleep(3)

print("Ollama server is starting in the background...")
print("Checking status...")

# Verify it's running by listing models
!ollama list


In [ ]:
cd scripts/model
for f in *.gguf; do
    # Get filename without extension for the model name
    model_name="${f%.gguf}"
    echo "Creating model entry for: $model_name"
    
    # Create a temporary Modelfile
    echo "FROM ./scripts/model/$f" > temp_modelfile
    
    # Register with Ollama
    ollama create "$model_name" -f temp_modelfile
    
    # Clean up
    rm temp_modelfile
done


In [2]:
import ollama
import json

In [ ]:
models = ["llama-3.2","qwen2.5","llama-3.2","deepseek1.5","smollm1.7","phi3.5","OpenELM1.1","ministral3.3","gemma3.4"]

In [4]:
system_prompt = """You are an expert computer science tutor specializing in Data Structures and Algorithms (DSA).

Your goal is to teach concepts clearly and effectively, not just provide answers.

Follow these principles strictly:

1. Explain Like a Teacher
- Start with intuition before formal definitions.
- Use simple language first, then refine with technical precision.
- Break complex ideas into small, digestible steps.

2. Structure Your Teaching
For every topic:
- Intuition / real-world analogy
- Formal explanation
- Step-by-step example
- Time and space complexity
- Common mistakes

3. Be Interactive
- Ask at least one question to check understanding.
- Offer a small practice problem after explanation.
- Adjust difficulty gradually.

4. Encourage Thinking
- Do NOT immediately give full solutions to problems.
- Provide hints first, then full solution only if necessary.

5. Use Clear Formatting
- Use bullet points, code blocks, and labeled sections.
- Highlight key insights.

6. Be Concise but Complete
- Avoid unnecessary verbosity.
- Ensure no critical step is skipped.

7. Adapt to the Student
- If the student seems confused, simplify further.
- If the student is comfortable, go deeper.

Your success is measured by how well the student understands, not by how much information you provide.
"""

In [ ]:
questions = ["Explain the difference between arrays and linked lists.",
    "Teach stack and queue with real-world analogies.",
    "Explain binary search including edge cases.",
    "Teach the two pointers technique with examples.",
    "Teach graph traversal (DFS and BFS).",
    "Explain dynamic programming using Fibonacci.",
    "Compare greedy algorithms vs dynamic programming.",
    "Explain recursion vs iteration.",
    "Teach backtracking using subsets or N-Queens.",
    "Explain binary tree traversals (DFS vs BFS)."]

In [ ]:
results = []
for model in models:
    for question in questions:
        try:
            response = ollama.chat(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": question}
                ]
            )

            answer = response["message"]["content"]

            results.append({
                "model": model,
                "question": question,
                "answer": answer
            })

            print(f"✅ Done: {model} | {question}")
            print(results)
        except Exception as e:
                print(f"❌ Error with {m}: {e}")
        with open("dsa_model_evaluation.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)

print("🎯 All results saved to dsa_model_evaluation.json")